## Análisis exploratorio de datos (EDA) para realizar la limpieza de datos


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, mean, stddev, min, max, desc
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType 

In [ ]:
#  Inicializar Spark
spark = SparkSession.builder \
    .appName("FinPlus_EDA") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


In [ ]:
# Cargar los datos desde el archivo CSV

DATA_PATH = r"C:\Users\olgar\Documents\GitHub\trabajo-big-data-olga-paula-y-adriana\data\BEHAVIOURAL_1.csv"

df1 = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'BEHAVIOURAL_1.csv'))

IllegalArgumentException: java.net.URISyntaxException: Relative path in absolute URI: C:%5CUsers%5Colgar%5CDocuments%5CGitHub%5Ctrabajo-big-data-olga-paula-y-adriana%5Cdata%5CBEHAVIOURAL_1.csv

In [ ]:

df_clients = spark.read.csv("data/CLIENTS.csv", header=True, inferSchema=True)
df_behaviour = spark.read.csv("data/BEHAVIOURAL.csv", header=True, inferSchema=True)

# Mostrar esquema y primeras filas
print("=== CLIENTES ===")
print(f"Filas: {df_clients.count()}, Columnas: {len(df_clients.columns)}")
df_clients.printSchema()
df_clients.show(5, truncate=False)

print("\n=== COMPORTAMIENTO ===")
print(f"Filas: {df_behaviour.count()}, Columnas: {len(df_behaviour.columns)}")
df_behaviour.printSchema()
df_behaviour.show(5, truncate=False)

In [ ]:
# Celda 5: Función para análisis de nulos y duplicados
def analyze_data_quality(df, df_name):
    print(f"\n=== CALIDAD DE DATOS: {df_name} ===")
    
    # 1. Nulos por columna
    print("\n1. Valores nulos por columna:")
    null_counts = df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) 
                           for c in df.columns])
    null_counts.show(vertical=True)
    
    # 2. Porcentaje de nulos
    total_rows = df.count()
    null_percentages = {}
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        percentage = (null_count / total_rows) * 100
        null_percentages[col_name] = percentage
        if percentage > 0:
            print(f"   {col_name}: {null_count} nulos ({percentage:.2f}%)")
    
    # 3. Duplicados
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"\n2. Filas duplicadas: {duplicate_count}")
    
    # 4. Tipos de datos problemáticos
    print("\n3. Columnas con posibles problemas de tipo:")
    for field in df.schema.fields:
        if str(field.dataType) == "StringType":
            # Verificar si columnas string podrían ser numéricas
            sample = df.select(col(field.name)).limit(10).toPandas()[field.name]
            numeric_like = sample.str.contains(r'^\d+\.?\d*$').any()
            if numeric_like:
                print(f"   {field.name}: String pero parece numérico")
    
    return null_percentages

# Ejecutar análisis
null_clients = analyze_data_quality(df_clients, "CLIENTES")
null_behaviour = analyze_data_quality(df_behaviour, "COMPORTAMIENTO")

In [ ]:
# Celda 6: Estadísticas para columnas numéricas
print("=== ESTADÍSTICAS DESCRIPTIVAS - CLIENTES ===")

# Seleccionar columnas numéricas
numeric_cols = [f.name for f in df_clients.schema.fields 
                if isinstance(f.dataType, (IntegerType, DoubleType))]

# Calcular estadísticas
for col_name in numeric_cols[:10]:  # Primeras 10 para no saturar
    stats = df_clients.select(
        mean(col(col_name)).alias("media"),
        stddev(col(col_name)).alias("std"),
        min(col(col_name)).alias("min"),
        max(col(col_name)).alias("max")
    ).collect()[0]
    
    print(f"\n{col_name}:")
    print(f"  Media: {stats['media']:.2f}")
    print(f"  Std: {stats['std']:.2f}")
    print(f"  Min: {stats['min']:.2f}")
    print(f"  Max: {stats['max']:.2f}")
    
    # Contar ceros o valores atípicos
    zero_count = df_clients.filter(col(col_name) == 0).count()
    if zero_count > 0:
        print(f"  Valores cero: {zero_count}")

In [ ]:
# Celda 7: Análisis de columnas categóricas
print("=== ANÁLISIS CATEGÓRICAS - CLIENTES ===")

categorical_cols = [f.name for f in df_clients.schema.fields 
                    if isinstance(f.dataType, StringType)]

for col_name in categorical_cols[:8]:  # Primeras 8
    print(f"\n{col_name}:")
    
    # Conteo de categorías
    value_counts = df_clients.groupBy(col_name).count().orderBy(desc("count"))
    
    print(f"  Número de categorías únicas: {value_counts.count()}")
    
    # Mostrar top 5 categorías
    if value_counts.count() > 10:
        print("  (Mostrando top 10 categorías)")
    
    for row in value_counts.take(10):
        print(f"    {row[col_name]}: {row['count']} ({row['count']/df_clients.count()*100:.1f}%)")